In [14]:
!pip install evaluate seqeval -q

In [2]:
!pip install evaluate seqeval -q

import os
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    DataCollatorForTokenClassification, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Configuration
MODEL_NAME = "distilbert-base-uncased"
DATA_PATH = "/kaggle/input/datasets/balloumoussa/train-data-v2/train.jsonl"
OUTPUT_DIR = "./models/SpriteStack_Model"

# Mapping labels to IDs (B-only as requested)
label_list = ["O", "B-OBJECT", "B-POSITION", "B-SCENE_TYPE", "B-COUNT"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# 2. Metrics Setup
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert to label names
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 3. Load and Preprocess Data
# Load dataset and create a split for validation metrics
raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset_split = raw_dataset.train_test_split(test_size=0.2)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens
            else:
                label_ids.append(label2id[label[word_idx]])
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the training and evaluation sets
train_dataset = dataset_split["train"].map(tokenize_and_align_labels, batched=True)
eval_dataset = dataset_split["test"].map(tokenize_and_align_labels, batched=True)

# 4. Initialize Model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",  # Corrected parameter name
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

# 7. Train and Save
print("🚀 Starting training...")
trainer.train()

# Final Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./models/SpriteStack_Model


In [3]:
import shutil
import os

# Create a fresh folder for just the final model
os.makedirs('/kaggle/working/final_model_slim', exist_ok=True)

# Define the files we actually need
files_to_copy = [
    'model.safetensors', 
    'config.json', 
    'tokenizer.json', 
    'tokenizer_config.json', 
    'special_tokens_map.json',
    'vocab.txt'
]

source_path = '/kaggle/working/models/SpriteStack_Model'

for file in files_to_copy:
    src = os.path.join(source_path, file)
    if os.path.exists(src):
        shutil.copy(src, '/kaggle/working/final_model_slim')

# Zip only the slim folder
shutil.make_archive('SpriteStack_Model_Slim', 'zip', '/kaggle/working/final_model_slim')

print("✅ Slim zip created! It should be much smaller (around 250MB).")

✅ Slim zip created! It should be much smaller (around 250MB).


In [4]:
from IPython.display import FileLink
import os

# Define the path to your slim zip file
slim_zip_path = '/kaggle/working/SpriteStack_Model_Slim.zip'

if os.path.exists(slim_zip_path):
    print("✅ Slim zip found! Click the link below to download:")
    display(FileLink(slim_zip_path))
else:
    print("❌ The slim zip file was not found. Make sure you ran the 'Slim Zip' script first.")

✅ Slim zip found! Click the link below to download:


/kaggle/working/SpriteStack_Model_Slim.zip